## **Aim**
To implement a program that identifies duplicate digital evidence files by comparing their SHA-256 hash values.

## **Algorithm**
**Step 1:** Import `hashlib`, `os`, `json`, and `collections.defaultdict`.

**Step 2:** Define a function `calculate_sha256(filepath)` to compute the SHA-256 hash of a file in chunks.

**Step 3:** Define `scan_directory(directory)` to recursively walk through a directory and compute hashes for all files.

**Step 4:** Use a dictionary mapping hash values to lists of file paths to group files by their hash.

**Step 5:** Identify groups with more than one file (duplicates).

**Step 6:** Display duplicate groups with file paths, sizes, and hash values.

**Step 7:** Optionally save the duplicate report to a JSON file.

In [1]:
import hashlib
import os
import json
import shutil
from collections import defaultdict

def calculate_sha256(filepath):
    sha256 = hashlib.sha256()
    try:
        with open(filepath, "rb") as f:
            for chunk in iter(lambda: f.read(8192), b""):
                sha256.update(chunk)
        return sha256.hexdigest()
    except Exception:
        return None

def scan_directory(directory):
    hash_map = defaultdict(list)
    file_count = 0
    
    for root, _, files in os.walk(directory):
        for file in files:
            filepath = os.path.join(root, file)
            file_hash = calculate_sha256(filepath)
            if file_hash:
                stat = os.stat(filepath)
                hash_map[file_hash].append({
                    "path": filepath,
                    "size": stat.st_size,
                    "mtime": stat.st_mtime
                })
                file_count += 1
    
    return hash_map, file_count

def find_duplicates(hash_map):
    duplicates = {h: files for h, files in hash_map.items() if len(files) > 1}
    return duplicates

def display_duplicates(duplicates):
    if not duplicates:
        print("No duplicate files found.")
        return
    
    print(f"\n{'='*60}")
    print(f"DUPLICATE FILES REPORT")
    print(f"{'='*60}")
    print(f"Total duplicate groups: {len(duplicates)}")
    
    total_dupes = sum(len(files) - 1 for files in duplicates.values())
    print(f"Total duplicate files: {total_dupes}")
    print(f"{'='*60}\n")
    
    for i, (file_hash, files) in enumerate(duplicates.items(), 1):
        print(f"Group {i} (SHA-256: {file_hash[:16]}...)")
        for j, file_info in enumerate(files):
            marker = "  [ORIGINAL]" if j == 0 else "  [DUPLICATE]"
            size_kb = file_info['size'] / 1024
            print(f"  {marker} {file_info['path']} ({size_kb:.2f} KB)")
        print()

def main():
    test_dir = "./duplicate_test"
    
    if os.path.exists(test_dir):
        shutil.rmtree(test_dir)
    os.makedirs(test_dir)
    os.makedirs(os.path.join(test_dir, "subdir"))
    
    # Create test files - some duplicates
    content1 = b"Evidence file A - Case 2026-001"
    content2 = b"Evidence file B - Case 2026-002"
    content3 = b"Unique file C"
    
    with open(os.path.join(test_dir, "evidence_a.txt"), "wb") as f:
        f.write(content1)
    with open(os.path.join(test_dir, "evidence_a_copy.txt"), "wb") as f:
        f.write(content1)  # Exact duplicate
    with open(os.path.join(test_dir, "subdir", "evidence_a_backup.txt"), "wb") as f:
        f.write(content1)  # Duplicate in subfolder
    with open(os.path.join(test_dir, "evidence_b.txt"), "wb") as f:
        f.write(content2)
    with open(os.path.join(test_dir, "evidence_b_dup.txt"), "wb") as f:
        f.write(content2)  # Duplicate of B
    with open(os.path.join(test_dir, "unique_c.txt"), "wb") as f:
        f.write(content3)
    
    print(f"Scanning directory: {test_dir}")
    hash_map, total_files = scan_directory(test_dir)
    print(f"Total files scanned: {total_files}")
    
    duplicates = find_duplicates(hash_map)
    display_duplicates(duplicates)
    
    # Save report
    report = {
        "scan_directory": test_dir,
        "total_files": total_files,
        "duplicate_groups": len(duplicates),
        "duplicates": {}
    }
    for h, files in duplicates.items():
        report["duplicates"][h] = [f["path"] for f in files]
    
    with open("duplicate_report.json", "w") as f:
        json.dump(report, f, indent=2)
    print("Duplicate report saved to duplicate_report.json")

if __name__ == "__main__":
    main()

Scanning directory: ./duplicate_test
Total files scanned: 6

DUPLICATE FILES REPORT
Total duplicate groups: 2
Total duplicate files: 3

Group 1 (SHA-256: 24d204c958a2ea78...)
    [ORIGINAL] ./duplicate_test/evidence_a.txt (0.03 KB)
    [DUPLICATE] ./duplicate_test/evidence_a_copy.txt (0.03 KB)
    [DUPLICATE] ./duplicate_test/subdir/evidence_a_backup.txt (0.03 KB)

Group 2 (SHA-256: 75a3a33a4ee3d032...)
    [ORIGINAL] ./duplicate_test/evidence_b.txt (0.03 KB)
    [DUPLICATE] ./duplicate_test/evidence_b_dup.txt (0.03 KB)

Duplicate report saved to duplicate_report.json


## **Result**
This the program successfully identifies duplicate digital evidence files by comparing their SHA-256 hash values.